<a href="https://colab.research.google.com/github/kushagrathisside/Image-Segmentation-Using-ViT-Finetuning/blob/main/RUGD_dataset_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget http://rugd.vision/data/RUGD_frames-with-annotations.zip

--2025-11-18 10:52:07--  http://rugd.vision/data/RUGD_frames-with-annotations.zip
Resolving rugd.vision (rugd.vision)... 16.15.188.127, 16.15.178.173, 16.15.196.168, ...
Connecting to rugd.vision (rugd.vision)|16.15.188.127|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5671298594 (5.3G) [application/zip]
Saving to: ‘RUGD_frames-with-annotations.zip’

RUGD_frames-with-an 100%[===================>]   5.28G  43.2MB/s    in 2m 11s  

2025-11-18 10:54:18 (41.3 MB/s) - ‘RUGD_frames-with-annotations.zip’ saved [5671298594/5671298594]



In [ ]:
!wget http://rugd.vision/data/RUGD_annotations.zip

--2025-11-18 10:56:31--  http://rugd.vision/data/RUGD_annotations.zip
Resolving rugd.vision (rugd.vision)... 52.217.168.125, 16.15.177.120, 16.15.203.68, ...
Connecting to rugd.vision (rugd.vision)|52.217.168.125|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 61129860 (58M) [application/zip]
Saving to: ‘RUGD_annotations.zip’

RUGD_annotations.zi 100%[===================>]  58.30M  27.5MB/s    in 2.1s    

2025-11-18 10:56:33 (27.5 MB/s) - ‘RUGD_annotations.zip’ saved [61129860/61129860]



In [ ]:
!unzip /content/RUGD_frames-with-annotations.zip

Streaming output truncated to the last 5000 lines.
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_01156.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_01896.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_00086.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_00096.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_00036.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_00131.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_01451.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_01636.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_02016.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_00571.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_01176.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_00216.png  
  inflating: RUGD_frames-with-annotations/trail-11/trail-11_00251.png  
  inflating: 

In [ ]:
!unzip /content/RUGD_annotations.zip

Streaming output truncated to the last 5000 lines.
  inflating: RUGD_annotations/trail/trail_00141.png  
  inflating: RUGD_annotations/trail/trail_03026.png  
  inflating: RUGD_annotations/trail/trail_00646.png  
  inflating: RUGD_annotations/trail/trail_00181.png  
  inflating: RUGD_annotations/trail/trail_02106.png  
  inflating: RUGD_annotations/trail/trail_00021.png  
  inflating: RUGD_annotations/trail/trail_02126.png  
  inflating: RUGD_annotations/trail/trail_02281.png  
  inflating: RUGD_annotations/trail/trail_03306.png  
  inflating: RUGD_annotations/trail/trail_00551.png  
  inflating: RUGD_annotations/trail/trail_02881.png  
  inflating: RUGD_annotations/trail/trail_01911.png  
  inflating: RUGD_annotations/trail/trail_00611.png  
  inflating: RUGD_annotations/trail/trail_02791.png  
  inflating: RUGD_annotations/trail/trail_01281.png  
  inflating: RUGD_annotations/trail/trail_02396.png  
  inflating: RUGD_annotations/trail/trail_02810.png  
  inflating: RUGD_annotations/t

In [ ]:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 32.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 re

In [ ]:
# @title 🚀 RUGD Dataset Patch Generator (Ground Truth Map)
import os
import cv2
import numpy as np
import random
import shutil
from glob import glob
from tqdm import tqdm
import albumentations as A
import json

In [ ]:

# @title CONFIGURATION UPDATE THIS PATH to match your Colab/Drive location
RAW_IMAGES_ROOT = '/content/RUGD_frames-with-annotations'
RAW_MASKS_ROOT = '/content/RUGD_annotations'
OUTPUT_DIR = '/content/RUGD_Processed_224'

# Patch Settings
PATCH_SIZE = 224
PATCHES_PER_IMAGE = 6   # Number of random crops per original frame
VAL_SPLIT = 0.1         # 10% data for validation

In [ ]:
# @title 1. GROUND TRUTH COLOR MAP
# [cite_start]Parsed directly from your uploaded 'RUGD_annotation-colormap.txt' [cite: 1]
# Format: {(R, G, B): Class_ID}
RUGD_COLOR_MAP = {
    (0, 0, 0): 0,          # void
    (108, 64, 20): 1,      # dirt
    (255, 229, 204): 2,    # sand
    (0, 102, 0): 3,        # grass
    (0, 255, 0): 4,        # tree
    (0, 153, 153): 5,      # pole
    (0, 128, 255): 6,      # water
    (0, 0, 255): 7,        # sky
    (255, 255, 0): 8,      # vehicle
    (255, 0, 127): 9,      # container/generic-object
    (64, 64, 64): 10,      # asphalt
    (255, 128, 0): 11,     # gravel
    (255, 0, 0): 12,       # building
    (153, 76, 0): 13,      # mulch
    (102, 102, 0): 14,     # rock-bed
    (102, 0, 0): 15,       # log
    (0, 255, 128): 16,     # bicycle
    (204, 153, 255): 17,   # person
    (102, 0, 204): 18,     # fence
    (255, 153, 204): 19,   # bush
    (0, 102, 102): 20,     # sign
    (153, 204, 255): 21,   # rock
    (102, 255, 255): 22,   # bridge
    (101, 101, 11): 23,    # concrete
    (114, 85, 47): 24      # picnic-table (Reconstructed from file split)
}

In [ ]:
def process_dataset():
    # A. Setup Output Directories
    if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
    for split in ['train', 'val']:
        os.makedirs(os.path.join(OUTPUT_DIR, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, split, 'masks'), exist_ok=True)

    # B. Find Files (Recursive Search for Subfolders)
    print(f"🔍 Searching for masks in {RAW_MASKS_ROOT}...")
    # Recursive glob to find files inside 'creek/', 'park-1/', etc.
    all_mask_paths = sorted(glob(os.path.join(RAW_MASKS_ROOT, "**", "*.png"), recursive=True))

    if len(all_mask_paths) == 0:
        print("❌ Error: No masks found! Check if you unzipped 'RUGD_annotations.zip'.")
        return

    print(f"📂 Found {len(all_mask_paths)} masks. Pairing with images...")

    # C. Define Sync Crop Pipeline
    aug = A.Compose([
        A.RandomCrop(width=PATCH_SIZE, height=PATCH_SIZE),
    ], additional_targets={'mask': 'mask'})

    processed_count = 0

    # D. Loop through Masks (Source of Truth)
    for mask_path in tqdm(all_mask_paths):
        # 1. Infer Image Path
        # mask_path example: /content/RUGD_annotations/park-1/park-1_00001.png
        # We need to find:   /content/RUGD_frames-with-annotations/park-1/park-1_00001.png

        # Extract relative path (e.g., "park-1/park-1_00001.png")
        rel_path = os.path.relpath(mask_path, RAW_MASKS_ROOT)
        img_path = os.path.join(RAW_IMAGES_ROOT, rel_path)

        # 2. Check if Image Exists
        if not os.path.exists(img_path):
            # Fallback: Try searching just by filename if folder structure differs
            filename = os.path.basename(mask_path)
            # This is slow, so only use if primary check fails.
            # Usually RUGD structure mirrors exactly.
            continue

        # 3. Load Data
        image = cv2.imread(img_path)
        if image is None: continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask_rgb = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)

        # 4. RGB -> ID Conversion
        h, w = mask_rgb.shape[:2]
        mask_id = np.zeros((h, w), dtype=np.uint8)

        for rgb_color, class_id in RUGD_COLOR_MAP.items():
            matches = np.all(mask_rgb == rgb_color, axis=-1)
            mask_id[matches] = class_id

        # 5. Generate Patches
        split = 'val' if random.random() < VAL_SPLIT else 'train'
        base_name = os.path.splitext(os.path.basename(mask_path))[0] # e.g. park-1_00001

        for i in range(PATCHES_PER_IMAGE):
            try:
                data = aug(image=image, mask=mask_id)
                crop_img = data['image']
                crop_mask = data['mask']

                # Save
                save_name = f"{base_name}_p{i}"
                cv2.imwrite(
                    os.path.join(OUTPUT_DIR, split, 'images', f"{save_name}.jpg"),
                    cv2.cvtColor(crop_img, cv2.COLOR_RGB2BGR)
                )
                cv2.imwrite(
                    os.path.join(OUTPUT_DIR, split, 'masks', f"{save_name}.png"),
                    crop_mask
                )
            except Exception:
                pass
        processed_count += 1

    # E. Finish
    print(f"\n✨ Processed {processed_count} image/mask pairs.")
    print(f"📦 Zipping dataset...")
    shutil.make_archive("/content/rugd_224_patches", 'zip', OUTPUT_DIR)
    print("✅ Ready! Download 'rugd_224_patches.zip' from the file browser.")

if __name__ == "__main__":
    process_dataset()

🔍 Searching for masks in /content/RUGD_annotations...
📂 Found 7436 masks. Pairing with images...


100%|██████████| 7436/7436 [44:36<00:00,  2.78it/s]



✨ Processed 7436 image/mask pairs.
📦 Zipping dataset...
✅ Ready! Download 'rugd_224_patches.zip' from the file browser.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

source_zip_path = '/content/rugd_224_patches.zip'
destination_drive_path = '/content/drive/MyDrive/rugd_224_patches.zip'

try:
    shutil.move(source_zip_path, destination_drive_path)
    print(f"Successfully moved '{source_zip_path}' to '{destination_drive_path}'")
except FileNotFoundError:
    print(f"Error: Source file '{source_zip_path}' not found.")
except Exception as e:
    print(f"An error occurred while moving the file: {e}")

Successfully moved '/content/rugd_224_patches.zip' to '/content/drive/MyDrive/rugd_224_patches.zip'


# for 518x518

In [ ]:
# Patch Settings
PATCH_SIZE2 = 518
PATCHES_PER_IMAGE2 = 4   # Number of random crops per original frame
OUTPUT_DIR2 = '/content/RUGD_Processed_518'

In [ ]:
def process_dataset():
    # A. Setup Output Directories
    if os.path.exists(OUTPUT_DIR2): shutil.rmtree(OUTPUT_DIR2)
    for split in ['train', 'val']:
        os.makedirs(os.path.join(OUTPUT_DIR2, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR2, split, 'masks'), exist_ok=True)

    # B. Find Files (Recursive Search for Subfolders)
    print(f"🔍 Searching for masks in {RAW_MASKS_ROOT}...")
    # Recursive glob to find files inside 'creek/', 'park-1/', etc.
    all_mask_paths = sorted(glob(os.path.join(RAW_MASKS_ROOT, "**", "*.png"), recursive=True))

    if len(all_mask_paths) == 0:
        print("❌ Error: No masks found! Check if you unzipped 'RUGD_annotations.zip'.")
        return

    print(f"📂 Found {len(all_mask_paths)} masks. Pairing with images...")

    # C. Define Sync Crop Pipeline
    aug = A.Compose([
        A.RandomCrop(width=PATCH_SIZE2, height=PATCH_SIZE2),
    ], additional_targets={'mask': 'mask'})

    processed_count = 0

    # D. Loop through Masks (Source of Truth)
    for mask_path in tqdm(all_mask_paths):
        # 1. Infer Image Path
        # mask_path example: /content/RUGD_annotations/park-1/park-1_00001.png
        # We need to find:   /content/RUGD_frames-with-annotations/park-1/park-1_00001.png

        # Extract relative path (e.g., "park-1/park-1_00001.png")
        rel_path = os.path.relpath(mask_path, RAW_MASKS_ROOT)
        img_path = os.path.join(RAW_IMAGES_ROOT, rel_path)

        # 2. Check if Image Exists
        if not os.path.exists(img_path):
            # Fallback: Try searching just by filename if folder structure differs
            filename = os.path.basename(mask_path)
            # This is slow, so only use if primary check fails.
            # Usually RUGD structure mirrors exactly.
            continue

        # 3. Load Data
        image = cv2.imread(img_path)
        if image is None: continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask_rgb = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)

        # 4. RGB -> ID Conversion
        h, w = mask_rgb.shape[:2]
        mask_id = np.zeros((h, w), dtype=np.uint8)

        for rgb_color, class_id in RUGD_COLOR_MAP.items():
            matches = np.all(mask_rgb == rgb_color, axis=-1)
            mask_id[matches] = class_id

        # 5. Generate Patches
        split = 'val' if random.random() < VAL_SPLIT else 'train'
        base_name = os.path.splitext(os.path.basename(mask_path))[0] # e.g. park-1_00001

        for i in range(PATCHES_PER_IMAGE2):
            try:
                data = aug(image=image, mask=mask_id)
                crop_img = data['image']
                crop_mask = data['mask']

                # Save
                save_name = f"{base_name}_p{i}"
                cv2.imwrite(
                    os.path.join(OUTPUT_DIR2, split, 'images', f"{save_name}.jpg"),
                    cv2.cvtColor(crop_img, cv2.COLOR_RGB2BGR)
                )
                cv2.imwrite(
                    os.path.join(OUTPUT_DIR2, split, 'masks', f"{save_name}.png"),
                    crop_mask
                )
            except Exception:
                pass
        processed_count += 1

    # E. Finish
    print(f"\n✨ Processed {processed_count} image/mask pairs.")
    print(f"📦 Zipping dataset...")
    shutil.make_archive("/content/rugd_518_patches", 'zip', OUTPUT_DIR2)
    print("✅ Ready! Download 'rugd_518_patches.zip' from the file browser.")

if __name__ == "__main__":
    process_dataset()

🔍 Searching for masks in /content/RUGD_annotations...
📂 Found 7436 masks. Pairing with images...


100%|██████████| 7436/7436 [47:48<00:00,  2.59it/s]



✨ Processed 7436 image/mask pairs.
📦 Zipping dataset...
✅ Ready! Download 'rugd_518_patches.zip' from the file browser.


In [ ]:
import shutil
import os

source_zip_path = '/content/rugd_518_patches.zip'
destination_drive_path = '/content/drive/MyDrive/rugd_518_patches.zip'

try:
    shutil.move(source_zip_path, destination_drive_path)
    print(f"Successfully moved '{source_zip_path}' to '{destination_drive_path}'")
except FileNotFoundError:
    print(f"Error: Source file '{source_zip_path}' not found.")
except Exception as e:
    print(f"An error occurred while moving the file: {e}")

Successfully moved '/content/rugd_518_patches.zip' to '/content/drive/MyDrive/rugd_518_patches.zip'
